In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import make_pipeline

import lime
import lime.lime_text

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import nltk

In [2]:
# Ganti dengan path yang sesuai di environment Kaggle Anda jika perlu
file_path = '/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv' 
df = pd.read_csv(file_path)

# Menampilkan 5 baris pertama untuk melihat sampel data
print("Contoh Data:")
display(df.head())

# Menampilkan informasi dasar tentang DataFrame (jumlah data, tipe kolom, etc.)
print("\nInformasi DataFrame:")
df.info()

Contoh Data:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Informasi DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [3]:
import nltk
import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# ==========================================================
# LANGKAH 1: Inisialisasi 'penanda' keberhasilan
# ==========================================================
download_berhasil = False

# ==========================================================
# LANGKAH 2: Unduh dan Verifikasi
# ==========================================================
print("Mencoba mengunduh komponen NLTK...")

try:
    # Mengunduh paket yang dibutuhkan
    nltk.download('punkt', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    
    # Memverifikasi sumber daya secara langsung
    nltk.data.find('taggers/averaged_perceptron_tagger')
    print("\n✅ Verifikasi Berhasil: Sumber daya 'averaged_perceptron_tagger' ditemukan.")
    # Jika berhasil, ubah 'penanda' menjadi True
    download_berhasil = True
    
except Exception as e:
    print(f"\n❌ Gagal: Terjadi masalah saat mengunduh atau memverifikasi sumber daya NLTK.")
    print(f"   Detail Error: {e}")
    print("   Silakan coba restart kernel (Kernel > Restart) dan jalankan lagi sel ini.")

Mencoba mengunduh komponen NLTK...

✅ Verifikasi Berhasil: Sumber daya 'averaged_perceptron_tagger' ditemukan.


In [4]:
# ==========================================================
# DEFINISI FUNGSI PREPROCESSING
# ==========================================================
def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

lemmatizer = WordNetLemmatizer()

def clean_text_advanced(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text) 
    text = text.lower()
    text = ' '.join([lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in nltk.word_tokenize(text)])
    return text

if download_berhasil:
    print("\nMemulai pra-pemrosesan lanjutan (lemmatization)...")
    # Terapkan fungsi ke DataFrame
    df['cleaned_review'] = df['review'].apply(clean_text_advanced)
    print("Pra-pemrosesan selesai.")
    print("\nContoh Review Setelah Dibersihkan (Advanced):\n", df['cleaned_review'][1])
else:
    print("\nPra-pemrosesan lanjutan DIBATALKAN karena sumber daya NLTK tidak siap.")



Memulai pra-pemrosesan lanjutan (lemmatization)...
Pra-pemrosesan selesai.

Contoh Review Setelah Dibersihkan (Advanced):
 a wonderful little production the film technique be very unassuming very oldtimebbc fashion and give a comfort and sometimes discomforting sense of realism to the entire piece the actor be extremely well chosen michael sheen not only have get all the polari but he have all the voice down pat too you can truly see the seamless edit guide by the reference to williams diary entry not only be it well worth the watch but it be a terrificly write and perform piece a masterful production about one of the great master of comedy and his life the realism really come home with the little thing the fantasy of the guard which rather than use the traditional dream technique remains solid then disappears it play on our knowledge and our sens particularly with the scene concern orton and halliwell and the set particularly of their flat with halliwells mural decorate every surface

In [5]:
# Inisialisasi LabelEncoder
le = LabelEncoder()

# Ubah kolom 'sentiment' menjadi format angka
df['sentiment_encoded'] = le.fit_transform(df['sentiment'])

# Tampilkan hasil untuk memastikan kolom baru sudah benar
display(df[['review', 'sentiment', 'sentiment_encoded']].head())

,review,sentiment,sentiment_encoded
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [6]:
# Inisialisasi TF-IDF Vectorizer dengan maksimal 5000 fitur/kata
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words='english')

# Buat matriks fitur (X) dan vektor target (y)
X = tfidf.fit_transform(df['cleaned_review']).toarray()
y = df['sentiment_encoded']

# Bagi data menjadi 80% data latih dan 20% data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Tampilkan ukuran dari setiap set data untuk verifikasi
print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test:", X_test.shape)
print("Ukuran y_train:", y_train.shape)
print("Ukuran y_test:", y_test.shape)

Ukuran X_train: (40000, 20000)
Ukuran X_test: (10000, 20000)
Ukuran y_train: (40000,)
Ukuran y_test: (10000,)


In [7]:
# --- Model 1: Multinomial Naive Bayes ---

# Inisialisasi model
nb_model = MultinomialNB()

# Latih model dengan data training
print("Melatih model Naive Bayes...")
nb_model.fit(X_train, y_train)
print("Pelatihan selesai.")

# Lakukan prediksi pada data test
y_pred_nb = nb_model.predict(X_test)

# Evaluasi performa model
accuracy_nb = accuracy_score(y_test, y_pred_nb)
report_nb = classification_report(y_test, y_pred_nb, target_names=['Negative', 'Positive'])

print(f"\nAkurasi Naive Bayes: {accuracy_nb * 100:.2f}%")
print("\nLaporan Klasifikasi Naive Bayes:")
print(report_nb)

Melatih model Naive Bayes...
Pelatihan selesai.

Akurasi Naive Bayes: 86.70%

Laporan Klasifikasi Naive Bayes:
              precision    recall  f1-score   support

    Negative       0.88      0.85      0.86      5000
    Positive       0.86      0.88      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [8]:
# --- Model 2: LinearSVC ---

# Inisialisasi model
# Perhatikan: LinearSVC tidak punya parameter 'kernel' atau 'gamma'
# dual=False direkomendasikan jika jumlah sampel > jumlah fitur, tapi kita coba auto dulu
linear_svm_model = LinearSVC(random_state=42, max_iter=1000)

# Latih model dengan data training
print("Melatih model LinearSVC...")
linear_svm_model.fit(X_train, y_train)
print("Pelatihan selesai.")

# Lakukan prediksi pada data test
y_pred_linear_svm = linear_svm_model.predict(X_test)

# Evaluasi performa model
accuracy_linear_svm = accuracy_score(y_test, y_pred_linear_svm)
report_linear_svm = classification_report(y_test, y_pred_linear_svm, target_names=['Negative', 'Positive'])

print(f"\nAkurasi LinearSVC: {accuracy_linear_svm * 100:.2f}%")
print("\nLaporan Klasifikasi LinearSVC:")
print(report_linear_svm)

Melatih model LinearSVC...
Pelatihan selesai.

Akurasi LinearSVC: 88.96%

Laporan Klasifikasi LinearSVC:
              precision    recall  f1-score   support

    Negative       0.90      0.88      0.89      5000
    Positive       0.88      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [9]:
# # --- Model 2: Support Vector Machine (dengan Verbose) ---

# svm_model = SVC(random_state=42, verbose=True)

# # Latih model dengan data training
# print("Melatih model SVM (dengan verbose)...")
# svm_model.fit(X_train, y_train)
# print("Pelatihan selesai.")

# # (Sisa kode evaluasi sama seperti sebelumnya)
# y_pred_svm = svm_model.predict(X_test)
# accuracy_svm = accuracy_score(y_test, y_pred_svm)
# report_svm = classification_report(y_test, y_pred_svm, target_names=['Negative', 'Positive'])

# print(f"\nAkurasi SVM: {accuracy_svm * 100:.2f}%")
# print("\nLaporan Klasifikasi SVM:")
# print(report_svm)

In [ ]:
# Modifikasi pada Sel 9
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

# Sesuaikan rentang parameter, fokus pada nilai C yang lebih kecil
param_grid_linear = {
    'C': [0.01, 0.1, 0.5, 1] 
}

# Tambahkan max_iter yang lebih tinggi untuk mengatasi ConvergenceWarning
grid_search_linear = GridSearchCV(LinearSVC(random_state=42, max_iter=3000), # <-- max_iter dinaikkan
                                  param_grid_linear, 
                                  refit=True, 
                                  verbose=2, 
                                  cv=3, 
                                  n_jobs=-1)

# Mulai pencarian pada data latih
print("Memulai pencarian hyperparameter terbaik untuk LinearSVC...")
grid_search_linear.fit(X_train, y_train)

# Tampilkan parameter terbaik yang ditemukan
print("\nParameter terbaik yang ditemukan:")
print(grid_search_linear.best_params_)

Memulai pencarian hyperparameter terbaik untuk LinearSVC...
Fitting 3 folds for each of 4 candidates, totalling 12 fits


In [ ]:
# Ambil model terbaik dari objek grid_search_linear
best_linear_svm_model = grid_search_linear.best_estimator_

# Lakukan prediksi pada data test
y_pred_best_linear_svm = best_linear_svm_model.predict(X_test)

# Evaluasi performa model yang sudah dioptimalkan
accuracy_best_linear_svm = accuracy_score(y_test, y_pred_best_linear_svm)
report_best_linear_svm = classification_report(y_test, y_pred_best_linear_svm, target_names=['Negative', 'Positive'])

print(f"\nAkurasi LinearSVC setelah optimasi: {accuracy_best_linear_svm * 100:.2f}%")
print("\nLaporan Klasifikasi LinearSVC (Optimized):")
print(report_best_linear_svm)

In [ ]:
# Ambil model terbaik dari grid search sebelumnya
# Ganti nama variabel ini jika Anda menamakannya berbeda
final_model = grid_search_linear.best_estimator_ 

# Ambil nama-nama fitur (kata dan n-gram) dari vectorizer
feature_names = tfidf.get_feature_names_out()

# Buat DataFrame untuk menampilkan kata dan bobotnya
coef_df = pd.DataFrame(final_model.coef_[0], index=feature_names, columns=['Coefficient'])

# Urutkan DataFrame berdasarkan bobot
sorted_coef_df = coef_df.sort_values(by='Coefficient', ascending=False)

# Tampilkan 20 kata teratas untuk sentimen positif dan negatif
print("Top 20 Kata untuk Sentimen Positif:")
display(sorted_coef_df.head(20))

print("\nTop 20 Kata untuk Sentimen Negatif:")
display(sorted_coef_df.tail(20))

In [ ]:
c = make_pipeline(tfidf, nb_model)

# Buat explainer untuk teks
explainer = lime.lime_text.LimeTextExplainer(class_names=['Negative', 'Positive'])

# Pilih satu contoh ulasan dari data uji untuk dianalisis
idx = 10 
text_instance = df.loc[y_test.index[idx], 'review']

# Minta LIME untuk menjelaskan prediksi pada contoh ini
explanation = explainer.explain_instance(text_instance, c.predict_proba, num_features=10)

# Tampilkan penjelasan di notebook (akan menyorot kata-kata)
explanation.show_in_notebook()

# Atau simpan ke file HTML
# explanation.save_to_file('explanation.html')

In [ ]:
# Ambil indeks acak dari data uji
random_indices = np.random.choice(y_test.index, 5, replace=False)

# Ambil data asli dan prediksi berdasarkan indeks acak
sample_df = df.loc[random_indices]
y_true_sample = y_test.loc[random_indices]

# Dapatkan prediksi dari model yang sudah dilatih
# Pastikan variabel model ini sudah ada dari sel-sel sebelumnya
y_pred_nb_sample = nb_model.predict(X_test[y_test.index.get_indexer(random_indices)])
y_pred_svm_sample = best_linear_svm_model.predict(X_test[y_test.index.get_indexer(random_indices)])


# Ubah label angka kembali ke teks (0/1 -> 'negative'/'positive')
y_true_text = le.inverse_transform(y_true_sample)
y_pred_nb_text = le.inverse_transform(y_pred_nb_sample)
y_pred_svm_text = le.inverse_transform(y_pred_svm_sample)

# Tampilkan hasilnya
for i in range(len(sample_df)):
    print(f"----- Ulasan Acak #{i+1} -----")
    print(f"TEKS       : {sample_df.iloc[i]['review'][:300]}...") # Tampilkan 300 karakter pertama
    print(f"LABEL ASLI : {y_true_text[i]}")
    print(f"TEBAKAN NB : {y_pred_nb_text[i]}")
    print(f"TEBAKAN SVM: {y_pred_svm_text[i]}")
    print("-" * (25 + len(str(i+1))))
    print("\\n")